In [1]:
from NN_functions import *
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import tensorflow as tf 

In [2]:
?simple_NN

Signature: simple_NN(X, y, seed=2, lr=0.1, hidden_size=5, max_iter=30, **kwargs)
Docstring:
X, y should be numpy arrays
W1_init: optional argument for initialized weight
W2_init: optional argument for initialized weight
File:      ~/Documents/GIT/MTDS/notebooks/NN_functions.py
Type:      function

In [20]:
# data prep
ames = pd.read_csv("../data/ames.csv")
ames_num = ames.select_dtypes(exclude=['object'])
yh = ames['Sale_Price']
Xh = ames_num.iloc[:,:-3] # selecting all but the last 3 columns
X_train, X_test, y_train, y_test = train_test_split(Xh, yh, test_size = 0.25, random_state = 45)
X_train_n = X_train.to_numpy()
y_train_n = y_train.to_numpy()
X_test_n = X_test.to_numpy()
y_test_n = y_test.to_numpy()

In [21]:
p1 = 5

In [22]:
# 3e4
W1_1, W2_1, J_1, u2_1 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 0.5e-13, max_iter = int(1e4))
print(J_1**0.5)
print(root_mean_squared_error(u2_1, y_train_n))

40472.64581352626
40472.64581352626


In [23]:
W1_2, W2_2, J_2, u2_2 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 0.5e-13, max_iter = int(1e4),
                                 W1_init = W1_1,
                                 W2_init = W2_1)
print(J_2**0.5)

40076.11782201403


In [24]:
W1_3, W2_3, J_3, u2_3 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 0.5e-13, max_iter = int(1e4),
                                 W1_init = W1_2,
                                 W2_init = W2_2)
print(J_3**0.5)

39570.033623801166


### Above, we have split these 30K iterations into 3 portions since we feed the previously trained weights into the next 10K.

In [26]:
y_pred = simple_NN_pred(X_test_n, W1_3, W2_3)
f"testing error is {root_mean_squared_error(y_pred, y_test)}"

'testing error is 43625.04589447899'

## adjusting learning rate
We know that the learning rate affects the training significantly, so we will go through some common optimizaiton techniques for having better (adaptive) learning rates.

In [11]:
def simple_NN_l1(X, y, seed = 2, hidden_size = 5, max_iter = 30, alpha = 0.5, lr = 1e-13, **kwargs):
    '''
    X, y should be numpy arrays
    W1_init: optional argument for initialized weight
    W2_init: optional argument for initialized weight
    '''
    n, p = X.shape
    # reshape y to a column
    y = y.reshape([-1, 1])

    # size (number of nodes) of hidden layer
    p1 = hidden_size
    
    # random initialization if initial weights not provided
    np.random.seed(seed)
    w_scale = np.sqrt(6/(p+1))
    W1_rand = np.random.rand(p, p1) * w_scale 
    W2_rand = np.random.rand(p1, 1) * w_scale

    W1 = kwargs.get('W1_init', W1_rand)
    W2 = kwargs.get('W2_init', W2_rand)

    v1 = 0
    v2 = 0
    for i in range(max_iter):
        # forward propargation (usually should be a for loop, iterate over how many hidden layers there are)
        u1, h, u2 = forward_prop(X, W1, W2) # u2 is supposed to be approx y
    
        # compute cost
        J = J_MSE(u2, y)
    
        # backward propagation for computing gradient (usually a for loop iterate over # of hidden layers)
        dW1, dW2 = backward_prop(X, y, u1, h, u2, W1, W2)
        
        # gradient descent with momentum
        v1 = alpha*v1 - lr*dW1
        v2 = alpha*v2 - lr*dW2
        W1 = W1 + v1
        W2 = W2 + v2

        # if alpha = 0, then this is the same as simple_NN()
        
    
    return W1, W2, J, u2

In [30]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n, y_train_n, hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = int(2e4),                                 
                                 )
print(J_m**0.5)

32285.171489054057


In [32]:
# test error?
y_pred = simple_NN_pred(X_test_n, W1_m, W2_m)
f"testing error for using momentum is {root_mean_squared_error(y_pred, y_test):.2f}"

'testing error for using momentum is 34955.01'

## Stochastic Gradient Descent (SGD)

In [37]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[:200,:], y_train_n[:200], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = 100,                                 
                                 )
print(J_m**0.5)

64892.14040390273


In [38]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[200:400,:], y_train_n[200:400], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

50185.41739878808


In [39]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[400:600,:], y_train_n[400:600], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

52100.58310298797


In [40]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[600:800,:], y_train_n[600:800], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

41201.74258487222


In [41]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[800:1000,:], y_train_n[800:1000], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

40584.736253393065
